In [3]:
import pandas as pd
import os
import time
import duckdb

In [5]:
parquet_path = "../data/raw/yellow_tripdata_2026-01.parquet"
csv_path = "../data/benchmark/yellow_tripdata_2026-01.csv"

In [2]:
# Prepare identical CSV and Parquet files
df = pd.read_parquet(parquet_path)
df.to_csv(csv_path, index=False)

In [9]:
# Measure pandas on CSV
start = time.perf_counter()

df = pd.read_csv(csv_path)

pandas_result = (
    df.groupby("PULocationID", dropna=False)["total_amount"]
      .mean()
      .reset_index(name="avg_val")
)

elapsed = time.perf_counter() - start

print("Method: pandas on CSV")
print(f"File size: {os.path.getsize(csv_path) / (1024**2):.2f} MB")
print(f"Query time: {elapsed:.4f} seconds")
print(f"Rows returned: {len(pandas_result)}")
print(pandas_result.head())

C:\Users\Laurent\AppData\Local\Temp\ipykernel_20608\2957107143.py:4: DtypeWarning: Columns (0: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Method: pandas on CSV
File size: 383.01 MB
Query time: 58.1172 seconds
Rows returned: 262
   PULocationID     avg_val
0             1  103.978128
1             2   64.603333
2             3   38.958391
3             4   26.684277
4             5   50.875000


In [7]:
# Measure DuckDB on CSV
query = """
SELECT
    PULocationID,
    AVG(total_amount) AS avg_val
FROM read_csv_auto(?)
GROUP BY PULocationID
ORDER BY PULocationID
"""

start = time.perf_counter()

duckdb_csv_result = duckdb.execute(
    query,
    [csv_path]
).fetchall()

elapsed = time.perf_counter() - start

print("Method: DuckDB on CSV")
print(f"File size: {os.path.getsize(csv_path) / (1024**2):.2f} MB")
print(f"Query time: {elapsed:.4f} seconds")
print(f"Rows returned: {len(duckdb_csv_result)}")
print(duckdb_csv_result[:5])

Method: DuckDB on CSV
File size: 383.01 MB
Query time: 5.6435 seconds
Rows returned: 262
[(1, 103.9781275720165), (2, 64.60333333333334), (3, 38.95839103869653), (4, 26.684276771203205), (5, 50.875)]


In [8]:
# Measure DuckDB on Parquet
query = """
SELECT
    PULocationID,
    AVG(total_amount) AS avg_val
FROM read_parquet(?)
GROUP BY PULocationID
ORDER BY PULocationID
"""

start = time.perf_counter()

duckdb_parquet_result = duckdb.execute(
    query,
    [parquet_path]
).fetchall()

elapsed = time.perf_counter() - start

print("Method: DuckDB on Parquet")
print(f"File size: {os.path.getsize(parquet_path) / (1024**2):.2f} MB")
print(f"Query time: {elapsed:.4f} seconds")
print(f"Rows returned: {len(duckdb_parquet_result)}")
print(duckdb_parquet_result[:5])

Method: DuckDB on Parquet
File size: 61.19 MB
Query time: 0.3514 seconds
Rows returned: 262
[(1, 103.9781275720165), (2, 64.60333333333334), (3, 38.95839103869654), (4, 26.684276771203415), (5, 50.875)]
